# 03 — CMF Search with AADT as Main Term

This template fits a **Crash Modification Factor (CMF)** model using
the `CMFExperimentBuilder`.  AADT (Annual Average Daily Traffic) is the
primary scaling term and is **always included** in the model.

**Two approaches are shown:**

| Approach | When to use |
|----------|-------------|
| **A. Full-flexibility JAX search** | Full role system (0–8), latent classes possible, `ModelConstraints` support |
| **B. Classic GA CMF search** | Original two-component AADT structure; fast; no LC |

**Two-component model structure:**
```
log(mu_i) = [alpha_0 + sum_k alpha_k * X_ki]           (Component A)
          + [beta_0  + sum_k beta_k  * X_ki] * log(AADT_i)  (Component B)

CMF_k (Component A) = exp(alpha_k)
CMF_k (Component B) = AADT_mean ^ beta_k
```

In [ ]:
import numpy as np
import pandas as pd

from metacountregressor import (
    CMFExperimentBuilder,
    ModelConstraints,
    load_example16_3_model_data,
    load_book_cmf_spec,
    describe_book_cmf_spec,
    get_help,
)

# Full CMF guide:
# get_help('cmf')

## 1 · Load data

In [ ]:
df = load_example16_3_model_data()

# Exposure offset (needed by both CMF builders)
exposure = df['LENGTH'] * df['AADT'] * 365 / 1e8
df['OFFSET'] = np.log(exposure.clip(lower=1e-9))

print(f'Rows: {len(df)}  |  AADT range: {df["AADT"].min():.0f}–{df["AADT"].max():.0f}')
df[['ID', 'FREQ', 'AADT', 'LENGTH', 'URB']].head(3)

## 2 · Define baseline and local variable sets

- **Baseline variables** enter Component A (independent of AADT)
- **Local variables** are multiplied by `log(AADT)` — Component B

In [ ]:
# Site characteristics — baseline (Component A)
baseline_vars = ['URB', 'ACCESS', 'GRADEBR', 'CURVES']

# AADT-interaction terms — local (Component B)
local_vars = ['CURVES', 'WIDTH']

cmf = CMFExperimentBuilder(
    df=df,
    y_col='FREQ',
    aadt_col='AADT',
    baseline_vars=baseline_vars,
    local_vars=local_vars,
)

## Approach A — Full-flexibility JAX search

`build_jax_count_evaluator()` forces `log(AADT)` to be **fixed and always
included** (role 1), then lets the full role system (0–8) apply to all
other variables.  Supports latent classes, random parameters, ZI, etc.

In [ ]:
c = (
    ModelConstraints()
    # AADT is forced fixed internally — no need to constrain it here
    # Road geometry vars cannot be ZI terms
    .no_zi('CURVES', 'WIDTH', 'GRADEBR', 'ACCESS', 'URB')
    # Allow random curvature effect (lognormal = positive direction)
    .allow_random('CURVES', distributions=['lognormal'])
    # Binary dummies — no taste variation
    .no_random('URB', 'GRADEBR')
)

builder_jax, evaluator_jax, meta = cmf.build_jax_count_evaluator(
    id_col='ID',
    offset_col='OFFSET',
    constraints=c,
    max_latent_classes=1,   # set to 2 to add LC capability
    R=200,
)

print('Search variables:', meta['search_vars'])
print('log(AADT) column:', meta['log_aadt_col'])

In [ ]:
result_jax = builder_jax.run(
    evaluator=evaluator_jax,
    algo='sa',
    max_iter=50,
    seed=42,
)

print('Best BIC:', result_jax.best_score)

## Approach B — Classic GA CMF search

The original genetic-algorithm search directly optimises the two-component
AADT structure.  Faster and more interpretable CMF output, but no latent
classes.

In [ ]:
# Run GA search
search_result = cmf.run_search(R=200)
print('GA search complete.')
print('Selected baseline vars:', search_result.selected_baseline)
print('Selected local vars:   ', search_result.selected_local)

In [ ]:
# Re-fit and compute standard errors
fit_result = cmf.fit_best_model(search_result, final_R=500)

# Print full CMF report
cmf.print_report(search_result, fit_result)

## 3 · Book specification — manual CMF model

In [ ]:
describe_book_cmf_spec()

In [ ]:
book_spec = load_book_cmf_spec()

# Build manual_spec from the book structure
manual_spec = cmf.make_manual_cmf_spec(
    baseline_fixed=book_spec['baseline_fixed'],
    local_fixed=book_spec['local_fixed'],
    baseline_random=book_spec['baseline_random'],
)
print('Manual spec:', manual_spec)

In [ ]:
# Re-estimate the book CMF model
book_fit = cmf.fit_manual_cmf_model(
    id_col='ID',
    manual_spec=manual_spec,
    model='nb',
    R=200,
)
print(book_fit)

## 4 · Reading CMF values

```
Component A coefficient alpha_k:
    CMF = exp(alpha_k)
    If alpha_k = 0.15  →  CMF = 1.16  (16 % more crashes)
    If alpha_k = -0.30 →  CMF = 0.74  (26 % fewer crashes)

Component B coefficient beta_k  (evaluated at mean AADT):
    CMF = AADT_mean ^ beta_k
    If beta_k = 0.05 and AADT_mean = 10000  →  CMF ≈ 1.26
```

**Next steps:** extend to 2-class LC CMF by setting `max_latent_classes=2`
in `build_jax_count_evaluator()` and adding `membership_only('FC_ENCODED')`
to the `ModelConstraints`.